In [1]:

%%capture --no-stderr

%pip install --quiet -U \
    llama_index \
    faiss_cpu \
    sentence-transformers \
    textwrap3 \
    python-dotenv \
    llama-index-embeddings-huggingface \
    llama-index-vector-stores-faiss \
    llama-index-llms-cohere



In [2]:
# Imports
import os
import time
import textwrap
from typing import Generator
from dotenv import load_dotenv

from sentence_transformers import CrossEncoder
from llama_index.core import (
    VectorStoreIndex,
    StorageContext,
    load_index_from_storage,
    PromptTemplate
)
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.faiss import FaissVectorStore
from llama_index.core.llms import ChatMessage
from llama_index.llms.mistralai import MistralAI
from IPython.display import display, Markdown, clear_output
import sys
from llama_index.llms.cohere import Cohere
from llama_index.core.llms import ChatMessage

/Users/louisgolding/Downloads/Raylow-main/.conda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:


load_dotenv()

class SustainabilityExpert:
    
    def __init__(self, use_streaming=True):
        # No try-except for imports since they'll be handled elsewhere
        
        self.vector_db_dir = "./faiss_semantic_chunking"
        self.embed_model = HuggingFaceEmbedding(
            model_name="sentence-transformers/all-MiniLM-L6-v2"
        )
        
        # Load FAISS vector store from persistence
        self.vector_store = FaissVectorStore.from_persist_dir(self.vector_db_dir)
        self.storage_context = StorageContext.from_defaults(
            vector_store=self.vector_store,
            persist_dir=self.vector_db_dir
        )
        
        # Load index
        self.index = load_index_from_storage(
            storage_context=self.storage_context,
            embed_model=self.embed_model  
        )
        
        # Configure retriever
        self.retriever = self.index.as_retriever(similarity_top_k=10)    
        self.re_ranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
        
        # Setup Mistral client
        self.client = MistralAI(
            api_key=os.environ["MISTRAL_API_KEY"],
            model="mistral-small-latest"
        )
        
        # Set model name for printing
        self.model = "mistral-small-latest"
        print(f"Using model: {self.model}")
        
        # Track token usage for reporting (estimated since Mistral may track differently)
        self.prompt_tokens = 0
        self.completion_tokens = 0
        
        # Flag for streaming responses
        self.use_streaming = use_streaming
        
        # Improved prompt template for better RAG performance
        self.qa_prompt_template = """\
You are an expert in sustainability regulations and ESG frameworks. Your task is to provide clear, accurate, and actionable information based on the provided context.

Context information:
{context}

User company data:
{user_data}

User question: {question}

Guidelines:
- Base your answer on both the information in the context provided and the user's company data
- Personalize your response to be relevant to the user's industry, company size, and sustainability goals
- If the context doesn't contain relevant information, say "I don't have enough information on this topic"
- Provide structured, detailed responses with bullet points or numbered lists when appropriate
- Include specific regulatory requirements, deadlines, and compliance steps if mentioned in the context
- Cite specific regulations or frameworks by name when they appear in the context
- Focus on practical implementation advice tailored to the user's company profile

Your answer:
"""

    def format_response(self, text: str, width: int = 80) -> str:
        """Format response with proper line wrapping"""
        return textwrap.fill(text, width=width)

    def rerank_documents(self, query: str, nodes: list) -> list:
        """Re-rank nodes using cross-encoder"""
        node_texts = [node.text for node in nodes]
        scores = self.re_ranker.predict([(query, text) for text in node_texts])
        scored_nodes = sorted(zip(nodes, scores), key=lambda x: x[1], reverse=True)
        return [node for node, _ in scored_nodes[:4]]  # Return top 4

    def _rewrite_query(self, question: str) -> str:
        """
        Method to rewrite the query for better retrieval performance.
        """
        try:
            # Create message for query rewriting
            messages = [
                ChatMessage(
                    role="user", 
                    content=f"Rewrite the following question to be more specific and detailed for retrieving information about sustainability regulations and ESG topics. Include specific regulatory frameworks, standards, or concepts that might be relevant:\n\nOriginal question: {question}\n\nRewritten question:"
                )
            ]
            
            # Get response from Mistral
            response = self.client.chat(
                messages=messages,
                temperature=0
            )
            
            # Estimate token usage (approximate)
            self.prompt_tokens += len(question) // 4
            self.completion_tokens += len(response.message.content) // 4
                
            rewritten_query = response.message.content.strip()
            
            # Display the rewritten query
            print(f"\n📝 Original query: \"{question}\"")
            print(f"🔄 Rewritten query: \"{rewritten_query}\"\n")
            
            return rewritten_query
        except Exception as e:
            print(f"\n⚠️ Error rewriting query: {str(e)}")
            print("Using original query instead.")
            return question

    def query(self, question: str, user_data: dict = None) -> str:
        """
        Query response with rewritten query enhancement and user data personalization
        
        Args:
            question: The user's question
            user_data: Optional user company data for personalized responses
        
        Returns:
            str: The response from the LLM
        """
        try:
            # Rewrite the query for better retrieval
            rewritten_question = self._rewrite_query(question)
            
            # Retrieve relevant nodes using the rewritten query
            nodes = self.retriever.retrieve(rewritten_question)
            print(f"Retrieved {len(nodes)} nodes")
            
            # Apply reranking
            reranked_nodes = self.rerank_documents(rewritten_question, nodes)
            print(f"Reranked to {len(reranked_nodes)} nodes")
            
            # Build context
            context = "\n\n".join([n.text for n in reranked_nodes])
            
            # Add user data to context if provided
            user_context = ""
            if user_data:
                user_context = f"\nUser Company Data:\n{json.dumps(user_data, indent=2)}\n"
                print(f"Added user company data to context")
            
            # Format the prompt with user data
            formatted_prompt = self.qa_prompt_template.format(
                context=context, 
                question=question,
                user_data=user_context
            )
            
            # Create message for answering
            messages = [
                ChatMessage(role="user", content=formatted_prompt)
            ]
            
            # Use model for getting response
            print(f"\nQuerying {self.model} for answer...")
            
            # Estimate prompt tokens
            self.prompt_tokens += len(formatted_prompt) // 4
            
            if self.use_streaming:
                # Use streaming for real-time response
                print("\n📝 Answer: ", end="")
                answer = ""
                response_stream = self.client.stream_chat(
                    messages=messages,
                    temperature=0
                )
                
                for response_delta in response_stream:
                    delta_text = response_delta.delta
                    print(delta_text, end="", flush=True)
                    answer += delta_text
                    # Roughly estimate completion tokens
                    self.completion_tokens += len(delta_text) // 4
                
                print("\n\n✅ Response streaming completed")
            else:
                # Use non-streaming for traditional response
                response = self.client.chat(
                    messages=messages,
                    temperature=0
                )
                
                # Estimate completion tokens
                self.completion_tokens += len(response.message.content) // 4
                
                # Print estimated token usage
                print(f"Estimated usage: ~{len(formatted_prompt) // 4} prompt tokens and ~{len(response.message.content) // 4} completion tokens")
                
                answer = response.message.content
                print("\n📝 Answer:")
                print(answer)
                print("\n✅ Received response from model")
            
            # Return the full answer
            return answer
                
        except Exception as e:
            print(f"\n❌ Error: {str(e)}")
            return f"An error occurred: {str(e)}"

    def report_token_usage(self):
        """Report the estimated token usage for this session"""
        print(f"\n📊 Estimated Token Usage Report:")
        print(f"   Prompt tokens: ~{self.prompt_tokens}")
        print(f"   Completion tokens: ~{self.completion_tokens}")
        print(f"   Total tokens: ~{self.prompt_tokens + self.completion_tokens}")
        print(f"   Note: Token counts are estimated as Mistral may count differently than OpenAI")


if __name__ == "__main__":
    expert = SustainabilityExpert(use_streaming=True)
    print("🌱 Sustainability Regulations Query System (using Mistral AI)")
    print("🔄 Streaming mode enabled - answers will appear word by word")
    print("Type 'q', 'exit', or 'quit' to end the session.\n")

    try:
        while True:
            question = input("Ask your question:\n> ").strip()
            if question.lower() in ('q', 'exit', 'quit'):
                # Report token usage before exiting
                expert.report_token_usage()
                print("\n👋 Goodbye!")
                break

            start_time = time.time()
            print("\n🔍 Searching regulations...")

            # Get the full response as a string
            response = expert.query(question)

            print(f"\n⏱️  Response time: {time.time() - start_time:.2f}s\n")
    except KeyboardInterrupt:
        # Show token usage even if user interrupts with Ctrl+C
        expert.report_token_usage()
        
        print("\n👋 Session terminated by user. Goodbye!")

Using model: mistral-small-latest
🌱 Sustainability Regulations Query System (using Mistral AI)
🔄 Streaming mode enabled - answers will appear word by word
Type 'q', 'exit', or 'quit' to end the session.


🔍 Searching regulations...

📝 Original query: "Hi, I don't know anything about CSRD reporting but my boss needs me to file a report for the company. I don't know where to start."
🔄 Rewritten query: "Hi, I need to get up to speed on Corporate Sustainability Reporting Directive (CSRD) reporting for our company, but I'm not familiar with the requirements. Could you provide detailed information on the following:

1. **CSRD Overview**: What is the CSRD, and how does it differ from the Non-Financial Reporting Directive (NFRD)?
2. **Scope and Applicability**: Which companies are required to comply with CSRD, and what is the timeline for implementation?
3. **Reporting Standards**: What are the European Sustainability Reporting Standards (ESRS), and how do they apply to CSRD reporting?
4. **Ke